# Part 3 — Predict no-shows
The test set is never used to select a model or threshold.

In [1]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
df=pd.read_csv('appointments_clean.csv')
X=df[['age','distance_km','fee_inr','lead_days','prior_no_shows','reminder_sent','booking_channel','visit_type']].copy()
X['reminder_sent']=X.reminder_sent.map({'Yes':1,'No':0})
X=pd.get_dummies(X,columns=['booking_channel','visit_type'])
y=df.no_show
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42)
print('Features',X.shape[1],'Train',len(X_train),'Test',len(X_test),'Test no-shows',y_test.sum())
print('Baseline accuracy',1-y_test.mean())

Features 11 Train 792 Test 198 Test no-shows 49
Baseline accuracy 0.7525252525252525


In [2]:
scaler=StandardScaler();X_train_s=scaler.fit_transform(X_train);X_test_s=scaler.transform(X_test)
lr=LogisticRegression(max_iter=1000).fit(X_train_s,y_train)
yp=lr.predict(X_test_s);pr=lr.predict_proba(X_test_s)[:,1]
print('Logistic:',accuracy_score(y_test,yp),precision_score(y_test,yp),recall_score(y_test,yp),f1_score(y_test,yp),roc_auc_score(y_test,pr))
print('Confusion matrix\n',confusion_matrix(y_test,yp))
print(pd.DataFrame({'feature':X.columns,'coefficient':lr.coef_[0]}))
i=X.columns.get_loc('reminder_sent');print('OR scaled unit',np.exp(lr.coef_[0][i]),'OR No-to-Yes',np.exp(lr.coef_[0][i]/scaler.scale_[i]))
print('These are associations, not causal effects.')

Logistic: 0.7878787878787878 0.6296296296296297 0.3469387755102041 0.4473684210526316 0.788933022873579
Confusion matrix
 [[139  10]
 [ 32  17]]
                    feature  coefficient
0                       age    -0.253880
1               distance_km     0.253448
2                   fee_inr     0.452761
3                 lead_days     0.614471
4            prior_no_shows     0.649612
5             reminder_sent    -0.580669
6       booking_channel_App    -0.046804
7     booking_channel_Phone     0.215881
8   booking_channel_Walk-In    -0.186364
9   visit_type_Consultation     0.015658
10     visit_type_Follow-up    -0.015658
OR scaled unit 0.559524141750829 OR No-to-Yes 0.3093980903124433
These are associations, not causal effects.


In [3]:
full_tree=DecisionTreeClassifier(random_state=42).fit(X_train,y_train)
small_tree=DecisionTreeClassifier(max_depth=3,random_state=42).fit(X_train,y_train)
for name,m in [('Unrestricted',full_tree),('Depth 3',small_tree)]:
 print(name,'train',m.score(X_train,y_train),'test',m.score(X_test,y_test),'depth',m.get_depth())
print('A large train-test gap indicates overfitting.')

Unrestricted train 1.0 test 0.6717171717171717 depth 15
Depth 3 train 0.821969696969697 test 0.7777777777777778 depth 3
A large train-test gap indicates overfitting.


In [4]:
rf=RandomForestClassifier(n_estimators=100,random_state=42).fit(X_train,y_train)
rfp=rf.predict(X_test);rfprob=rf.predict_proba(X_test)[:,1]
print('Forest',accuracy_score(y_test,rfp),recall_score(y_test,rfp),roc_auc_score(y_test,rfprob))
print('Top features',pd.Series(rf.feature_importances_,index=X.columns).sort_values(ascending=False).head(3))
comparison=pd.DataFrame([
 ['Logistic regression',accuracy_score(y_test,yp),recall_score(y_test,yp),roc_auc_score(y_test,pr)],
 ['Depth-3 tree',accuracy_score(y_test,small_tree.predict(X_test)),recall_score(y_test,small_tree.predict(X_test)),roc_auc_score(y_test,small_tree.predict_proba(X_test)[:,1])],
 ['Random forest',accuracy_score(y_test,rfp),recall_score(y_test,rfp),roc_auc_score(y_test,rfprob)]],columns=['model','accuracy','recall','roc_auc'])
print(comparison.to_string(index=False))
print('Comparison is descriptive; formal model selection below uses training folds only.')

Forest 0.7474747474747475 0.30612244897959184 0.7057252431173813
Top features lead_days      0.219905
age            0.196788
distance_km    0.194739
dtype: float64
              model  accuracy   recall  roc_auc
Logistic regression  0.787879 0.346939 0.788933
       Depth-3 tree  0.777778 0.224490 0.699699
      Random forest  0.747475 0.306122 0.705725
Comparison is descriptive; formal model selection below uses training folds only.


In [5]:
grid=GridSearchCV(RandomForestClassifier(random_state=42),{'n_estimators':[100,200],'max_depth':[3,5,8]},cv=5,scoring='roc_auc',n_jobs=-1)
grid.fit(X_train,y_train)
lr_pipe=Pipeline([('scaler',StandardScaler()),('model',LogisticRegression(max_iter=1000))])
candidates={'Logistic regression':lr_pipe,'Depth-3 tree':DecisionTreeClassifier(max_depth=3,random_state=42),'Random forest':RandomForestClassifier(n_estimators=100,random_state=42),'Tuned forest':grid.best_estimator_}
cv_scores={name:cross_val_score(model,X_train,y_train,cv=5,scoring='roc_auc').mean() for name,model in candidates.items() if name!='Tuned forest'}
cv_scores['Tuned forest']=grid.best_score_
print('Grid best',grid.best_params_,grid.best_score_,'CV scores',cv_scores)
winner_name=max(cv_scores,key=cv_scores.get);winner=candidates[winner_name]
print('Chosen on training folds:',winner_name)
oof=cross_val_predict(winner,X_train,y_train,cv=5,method='predict_proba')[:,1]
threshold_rows=[]
for threshold in [.5,.35]:
 flags=(oof>=threshold).astype(int)
 threshold_rows.append([threshold,precision_score(y_train,flags,zero_division=0),recall_score(y_train,flags),f1_score(y_train,flags),int(flags.sum())])
threshold_table=pd.DataFrame(threshold_rows,columns=['threshold','precision','recall','f1','flagged'])
print(threshold_table)
chosen_threshold=float(threshold_table.loc[threshold_table.f1.idxmax(),'threshold'])
print('Chosen threshold',chosen_threshold,'(recall matters because a missed appointment leaves an empty slot)')
winner.fit(X_train,y_train)
test_prob=winner.predict_proba(X_test)[:,1]
test_flag=(test_prob>=chosen_threshold).astype(int)
print('FINAL TEST precision',precision_score(y_test,test_flag,zero_division=0),'recall',recall_score(y_test,test_flag),'flagged',test_flag.sum(),'AUC',roc_auc_score(y_test,test_prob))
output=pd.DataFrame({'appointment_id':df.loc[X_test.index,'appointment_id'].to_numpy(),'no_show':y_test.to_numpy(),'risk_prob':test_prob,'flag':test_flag})
output.to_csv('test_risk_scores.csv',index=False)
print('In practice choose threshold using evening call capacity versus empty-slot cost.')

Grid best {'max_depth': 3, 'n_estimators': 200} 0.7673242251739405 CV scores {'Logistic regression': np.float64(0.7737139899948249), 'Depth-3 tree': np.float64(0.6784494968661952), 'Random forest': np.float64(0.7466024782933701), 'Tuned forest': np.float64(0.7673242251739405)}
Chosen on training folds: Logistic regression
   threshold  precision    recall        f1  flagged
0       0.50   0.575342  0.248521  0.347107       73
1       0.35   0.512987  0.467456  0.489164      154
Chosen threshold 0.35 (recall matters because a missed appointment leaves an empty slot)
FINAL TEST precision 0.5714285714285714 recall 0.5714285714285714 flagged 49 AUC 0.788933022873579
In practice choose threshold using evening call capacity versus empty-slot cost.
